## Intuition

Suppose you have 3 documents:
```
Doc1: "I love machine learning and artificial intelligence"
Doc2: "Machine learning is used in recommendation systems"
Doc3: "Dogs love playing in parks"
```
Query:
```
machine learning
```

You would expect:
```
Doc1 → relevant
Doc2 → very relevant
Doc3 → irrelevant
```

### BM25 scores each document based on
* **Term Frequency (TF)**: How many times query words appear.
* **Inverse Document Frequency (IDF)**: Rare words are more important.
* **Document Length Normalization**: Longer documents don't get unfair advantage.

### BM25 Formula

For each query term:
$$\text{Score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \times \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$


Where:
| Symbol | Meaning |
| :--- | :--- |
| $f(q_i, D)$ | Term frequency of $q_i$ in document $D$ |
| $\text{IDF}(q_i)$ | Inverse document frequency of term $q_i$ |
| $\|D\|$ | Document length (number of words) |
| $\text{avgdl}$ | Average document length across the collection |
| $k_1$ | Tuning parameter for term frequency saturation (usually 1.2 to 2.0) |
| $b$ | Tuning parameter for document length normalization (usually 0.75) |


In [3]:
# Tokenize
docs = [
    "machine learning is fun".split(),
    "machine learning machine learning".split(),
    "dogs and cats".split()
]

In [14]:
# Compute IDF
from math import log
import json

N = len(docs)

df = {}

for doc in docs:
    for word in set(doc):
        df[word] = df.get(word, 0) + 1

idf = {}

for word, freq in df.items():
    idf[word] = log(
        (N - freq + 0.5) /
        (freq + 0.5) + 1
    )

print(json.dumps(idf, indent=4))

{
    "fun": 0.9808292530117263,
    "learning": 0.47000362924573563,
    "machine": 0.47000362924573563,
    "is": 0.9808292530117263,
    "cats": 0.9808292530117263,
    "and": 0.9808292530117263,
    "dogs": 0.9808292530117263
}


In [5]:
# BM25 Score Function
from collections import Counter

k1 = 1.5
b = 0.75

avgdl = sum(len(d) for d in docs) / N

def bm25_score(query, doc):
    score = 0

    doc_len = len(doc)
    freq = Counter(doc)

    for term in query:

        if term not in freq:
            continue

        tf = freq[term]

        numerator = tf * (k1 + 1)

        denominator = (
            tf +
            k1 * (
                1 - b +
                b * doc_len / avgdl
            )
        )

        score += idf.get(term, 0) * numerator / denominator

    return score

In [13]:
# Rank Documents
import json

query = "machine learning".split()

scores = []

for i, doc in enumerate(docs):
    score = bm25_score(query, doc)
    scores.append((i, score))

scores.sort(
    key=lambda x: x[1],
    reverse=True
)

print(json.dumps(scores, indent=4))

[
    [
        1,
        1.3047419360764898
    ],
    [
        0,
        0.9030637417821993
    ],
    [
        2,
        0
    ]
]


Doc2 ranks highest